<a href="https://colab.research.google.com/github/naman-0804/learning/blob/langgraph/tools_react_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
## Gemini API Setup

To use the Gemini API, you'll need an API key. If you don't already have one, create a key in Google AI Studio. In Colab, add the key to the secrets manager under the "🔑" in the left panel. Give it the name `GOOGLE_API_KEY`.
```

In [1]:
# Import the Python SDK
import google.generativeai as genai
# Used to securely store your API key
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


```markdown
Now, let's initialize the Generative Model, which we will refer to as `llm`.
```

In [2]:
pip install langchain-google-genai langchain-core --quiet

In [13]:
# Initialize the Gemini API model
#!pip install langchain_google_genai
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0.0, google_api_key=GOOGLE_API_KEY)

In [14]:
import google.generativeai as genai

#Creating a tool
def add(a:int,b:int)->int:
  """
  Add two integers.
  Args:
      a (int): The first integer.
      b (int): The second integer.
  Returns:
      int: The sum of a and b.
  """
  return a+b
def sub(a:int,b:int)->int:
  """
  Subtract two integers.
  Args:
      a (int): The first integer.
      b (int): The second integer.
  Returns:
      int: The difference between a and b.
  """
  return a-b

In [17]:
from langchain_core.messages import HumanMessage

# Now binding to LLM so that LLM can use
llm_with_tools=llm.bind_tools([add, sub])
tool_call=llm_with_tools.invoke([HumanMessage(content=f"What is 5 plus 2 and 7 minus 1" , name ='Naman')])

In [18]:
tool_call.tool_calls

[{'name': 'add',
  'args': {'b': 2, 'a': 5},
  'id': 'call_1859536',
  'type': 'tool_call'},
 {'name': 'sub',
  'args': {'b': 1, 'a': 7},
  'id': 'call_1859537',
  'type': 'tool_call'}]

In [26]:
from langgraph.prebuilt import create_react_agent

# Simple way to create the agent with tools
agent_executor = create_react_agent(llm, tools=[add, sub])

# Run it and show the output
query = "What is 5 plus 2 and 7 minus 1?"
result = agent_executor.invoke({"messages": [("human", query)]})

# Display the final response
print("Final Answer:", result["messages"][-1].content)

/tmp/ipykernel_2899/3517849442.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools=[add, sub])


Final Answer: [{'type': 'text', 'text': '5 plus 2 is 7, and 7 minus 1 is 6.', 'extras': {'signature': 'EmgKZgERTTIPkCUluJ1+nxyLrn1QMKvd3UYF57QQ1bPlB9pxwTvs7dUcLx9an68GRP5BeDzkstmWVzqPL3rD7NjavyUfwQoYmNWkdHGWlMCd9u9iUNVos609mR8U1E3LAHP741YywN+LIw=='}}]


```markdown
### Manual LangGraph Implementation
This implementation shows how the state is passed between the model and the tools manually.
```

In [27]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

# 1. Define the State
class State(TypedDict):
    messages: Annotated[list, add_messages]

# 2. Define the graph nodes
def call_model(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# Define the Tool Node
tools = [add, sub]
tool_node = ToolNode(tools)

# 3. Define the routing logic
def should_continue(state: State):
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return END

# 4. Construct the Graph
workflow = StateGraph(State)

workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue)
workflow.add_edge("tools", "agent")

# 5. Compile and Run
app = workflow.compile()

final_output = app.invoke({"messages": [("human", "What is 5 + 2 and 7 - 1?")]})
for message in final_output["messages"]:
    message.pretty_print()

================================ Human Message =================================

What is 5 + 2 and 7 - 1?
================================== Ai Message ==================================

[]
Tool Calls:
  add (call_1310639)
 Call ID: call_1310639
  Args:
    a: 5
    b: 2
  sub (call_1310640)
 Call ID: call_1310640
  Args:
    a: 7
    b: 1
================================= Tool Message =================================
Name: add

7
================================= Tool Message =================================
Name: sub

6
================================== Ai Message ==================================

[{'type': 'text', 'text': '5 + 2 is 7, and 7 - 1 is 6.', 'extras': {'signature': 'EmgKZgERTTIPR5NEG3rUrJLZCf2X7dWmTSLcio/VHDpUCYH65GKALecMOtccdBDYiFM2zujKzXiQYybFfM9KWH9prqjvl/rEsAzjmiBQ05V3RAxHAT8Jj0vCf13Qv/oMkN4WzRL2wAXrTg=='}}]
